# ADC suitability score simulation

This notebook contains only model training and ADC score prediction for a new input CSV.

In [1]:
import re
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
from abnumber import Chain
from Bio.SeqUtils.ProtParam import ProteinAnalysis
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression, Ridge
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

pd.set_option("display.max_columns", None)
warnings.filterwarnings("ignore", message="Use Chain.multiple_domains")

if (Path.cwd() / "dataset").is_dir():
    PROJECT_ROOT = Path.cwd()
    OUTPUT_DIR = PROJECT_ROOT / "simulation"
elif (Path.cwd().parent / "dataset").is_dir():
    PROJECT_ROOT = Path.cwd().parent
    OUTPUT_DIR = Path.cwd()
else:
    raise FileNotFoundError("Could not find the dataset directory.")

DATASET_DIR = PROJECT_ROOT / "dataset"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Change this filename when testing a new input CSV.
TEST_INPUT_CSV = DATASET_DIR / "randomtest.CSV"
PREDICTION_OUTPUT_CSV = OUTPUT_DIR / "simulation_adc_scores.csv"

## CDR3 feature extraction

In [2]:
MODEL_CDR_NAMES = ["HCDR3", "LCDR3"]
MODEL_PROPERTY_SUFFIXES = [
    "pI",
    "net_charge_pH7.4",
    "net_charge_pH6.0",
    "hydrophobicity_GRAVY",
]
final_feature_cols = ["Subtype"] + [
    f"{cdr}_{suffix}"
    for cdr in MODEL_CDR_NAMES
    for suffix in MODEL_PROPERTY_SUFFIXES
]

def clean_sequence(sequence):
    return re.sub(r"\s+", "", str(sequence)).upper()

def valid_sequence(sequence):
    return bool(re.fullmatch(r"[ACDEFGHIKLMNPQRSTVWY]+", clean_sequence(sequence)))

def calculate_cdr3_features(heavy_sequence, light_sequence):
    result = {"sequence_error": None}
    try:
        heavy = clean_sequence(heavy_sequence)
        light = clean_sequence(light_sequence)
        if not valid_sequence(heavy) or not valid_sequence(light):
            raise ValueError("invalid heavy or light chain sequence")

        chains = {
            "HCDR3": Chain(heavy, scheme="imgt").cdr3_seq,
            "LCDR3": Chain(light, scheme="imgt").cdr3_seq,
        }
        for cdr_name, cdr_sequence in chains.items():
            protein = ProteinAnalysis(str(cdr_sequence))
            result.update({
                f"{cdr_name}_pI": protein.isoelectric_point(),
                f"{cdr_name}_net_charge_pH7.4": protein.charge_at_pH(7.4),
                f"{cdr_name}_net_charge_pH6.0": protein.charge_at_pH(6.0),
                f"{cdr_name}_hydrophobicity_GRAVY": protein.gravy(),
            })
    except Exception as error:
        result["sequence_error"] = str(error)
    return result

def add_model_features(data):
    required = ["Subtype", "Heavy Chain Sequence", "Light Chain Sequence"]
    missing = [column for column in required if column not in data.columns]
    if missing:
        raise ValueError(f"Missing required columns: {missing}")

    calculated = data.apply(
        lambda row: calculate_cdr3_features(
            row["Heavy Chain Sequence"], row["Light Chain Sequence"]
        ),
        axis=1,
    )
    return pd.concat(
        [data.reset_index(drop=True), pd.DataFrame(calculated.tolist())],
        axis=1,
    )

def read_csv_with_fallback(path):
    for encoding in ["utf-8-sig", "cp949"]:
        try:
            return pd.read_csv(path, encoding=encoding).dropna(how="all")
        except UnicodeDecodeError:
            continue
    raise UnicodeDecodeError("csv", b"", 0, 1, f"Unsupported encoding: {path}")

## Training data and model fitting

In [3]:
def normalize_status(value):
    return re.sub(r"\s+", " ", str(value).strip().lower())

status_score_map = {
    "approved": 10.0,
    "phase 3": 9.0,
    "phase 3(terminated)": 8.0,
    "phase2": 6.5,
    "phase2(terminated)": 5.0,
    "phase 1": 2.5,
    "phase 1(terminated)": 1.0,
}
model_status_map = {
    "phase 2/3": "phase 3",
    "phase 1/2": "phase 1",
}

training_frames = []
for filename in ["approved.CSV", "phase3.CSV", "phase2.CSV", "phase1.CSV"]:
    frame = read_csv_with_fallback(DATASET_DIR / filename)
    training_frames.append(add_model_features(frame))

training_data = pd.concat(training_frames, ignore_index=True)
training_data["actual_status"] = training_data["ADC status"].map(normalize_status)
training_data["model_status"] = training_data["actual_status"].replace(model_status_map)
training_data["target_score"] = training_data["model_status"].map(status_score_map)
training_data = training_data[
    training_data["target_score"].notna()
    & training_data["sequence_error"].isna()
].reset_index(drop=True)

X = training_data[final_feature_cols].copy()
y_score = training_data["target_score"].astype(float)
y_approved_phase3 = training_data["model_status"].isin(["approved", "phase 3"]).astype(int)

STATUS_REGRESSION_WEIGHT = 0.90
APPROVED_PHASE3_RANKING_WEIGHT = 0.10

def make_preprocessor():
    numeric_features = [feature for feature in final_feature_cols if feature != "Subtype"]
    return ColumnTransformer([
        ("numeric", Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler()),
        ]), numeric_features),
        ("categorical", Pipeline([
            ("imputer", SimpleImputer(strategy="constant", fill_value="Unknown")),
            ("onehot", OneHotEncoder(handle_unknown="ignore")),
        ]), ["Subtype"]),
    ])

status_model = Pipeline([
    ("preprocessor", make_preprocessor()),
    ("model", Ridge(alpha=0.1)),
]).fit(X, y_score)

ranking_model = Pipeline([
    ("preprocessor", make_preprocessor()),
    ("model", LogisticRegression(
        C=0.3, class_weight="balanced", max_iter=3000, random_state=42
    )),
]).fit(X, y_approved_phase3)

print(f"Training complete: {len(X)} antibodies, {len(final_feature_cols)} features")

Training complete: 100 antibodies, 9 features


## ADC score prediction

In [4]:
def build_model_input(feature_data):
    model_input = feature_data.copy()
    model_input["Subtype"] = (
        model_input["Subtype"]
        .fillna("Unknown")
        .astype(str)
        .str.replace(r"\s+", " ", regex=True)
        .str.strip()
        .replace("", "Unknown")
    )
    for feature in final_feature_cols:
        if feature not in model_input.columns:
            model_input[feature] = np.nan
    return model_input[final_feature_cols]

def predict_adc_score(feature_data):
    model_input = build_model_input(feature_data)
    status_score = np.clip(status_model.predict(model_input), 1.0, 10.0)
    approved_phase3_probability = ranking_model.predict_proba(model_input)[:, 1]
    ranking_score = 1.0 + 9.0 * approved_phase3_probability
    adc_score = np.clip(
        STATUS_REGRESSION_WEIGHT * status_score
        + APPROVED_PHASE3_RANKING_WEIGHT * ranking_score,
        1.0,
        10.0,
    )
    return np.round(adc_score, 3)

In [5]:
test_data = read_csv_with_fallback(TEST_INPUT_CSV).reset_index(drop=True)
test_features = add_model_features(test_data)

test_features["adc_score"] = np.nan
valid_rows = test_features["sequence_error"].isna()
test_features.loc[valid_rows, "adc_score"] = predict_adc_score(
    test_features.loc[valid_rows]
)

identity_columns = [
    column
    for column in ["ADC name", "Antibody name", "Subtype"]
    if column in test_features.columns
]
prediction_output = test_features[identity_columns + ["adc_score"]].copy()
prediction_output.to_csv(PREDICTION_OUTPUT_CSV, index=False, encoding="utf-8-sig")

print(f"Input: {TEST_INPUT_CSV}")
print(f"Output: {PREDICTION_OUTPUT_CSV}")
prediction_output

Input: /home/iconic/urp/dataset/randomtest.CSV
Output: /home/iconic/urp/simulation/simulation_adc_scores.csv


,ADC name,Antibody name,Subtype,adc_score
0,1.0,5B2,Chimeric IgG1-kappa,6.218
1,2.0,3C2,Chimeric IgG1-kappa,6.184
2,3.0,7H9,Chimeric IgG1-kappa,6.184
